In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.lines import Line2D

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "utils").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "utils").exists():
    raise RuntimeError("Run this notebook from inside the scotland repository.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from utils import new_figure, add_panel_labels, WIDTHS, CONTEXTS, CLADES  # noqa: E402

from sse_detection.lib.palettes import CANDIDATE_COLOR, BACKGROUND_COLOR  # noqa: E402
from sse_detection.lib.sse_diagnostics import main as sse_diagnostics  # noqa: E402

%load_ext autoreload
%autoreload 2

out_path = PROJECT_ROOT / "sse_detection/results/sse_outputs"
cluster_table = pd.read_parquet(out_path / "cluster_table.parquet")
edge_table = pd.read_parquet(out_path / "edge_table.parquet")

In [ ]:
print(cluster_table["candidate_tier"].value_counts())

In [ ]:
cluster_table.groupby("candidate_tier")["cluster_size"].describe().T

In [ ]:
sse_diagnostics(cluster_table)

In [ ]:
DIMENSION = ["burst", "burden"]  # expands to <axis>_score, _n, _upper_p
GRAY = "#6C6F73"
DIMENSION_LABELS = {
    "burst": "Local burst",
    "burden": "Onward burden",
}
DIMENSION_CANDIDATE_TIERS = {
    "burst": ("high_priority_burst", "high_priority_both_axes"),
    "burden": ("high_priority_burden", "high_priority_both_axes"),
}
DEFAULT_CANDIDATE_TIERS = HIGH_PRIORITY_TIERS = (
    "high_priority_both_axes",
    "high_priority_burst",
    "high_priority_burden",
)
INELIGIBLE = "size_ineligible"
SIZE_COL = "cluster_size"
TIER_COL = "candidate_tier"


def dimension_label(axis: str) -> str:
    return DIMENSION_LABELS.get(axis, axis.replace("_", " ").title())


def dimension_cols(axis: str) -> dict[str, str]:
    return {
        "score": f"{axis}_score",
        "n": f"{axis}_score_n",
        "p": f"{axis}_score_upper_p",
        "null_mean": f"{axis}_score_null_mean",
        "null_sd": f"{axis}_score_null_sd",
        "null_z": f"{axis}_score_null_z",
    }


def _as_tier_tuple(
    candidate_tiers: str | tuple[str, ...] | list[str] | set[str],
) -> tuple[str, ...]:
    return (
        (candidate_tiers,)
        if isinstance(candidate_tiers, str)
        else tuple(candidate_tiers)
    )


def _candidate_mask(
    df: pd.DataFrame,
    *,
    tier_col: str = TIER_COL,
    candidate_tiers: str
    | tuple[str, ...]
    | list[str]
    | set[str] = DEFAULT_CANDIDATE_TIERS,
) -> pd.Series:
    return df[tier_col].isin(_as_tier_tuple(candidate_tiers))


def legend_candidate_background() -> list[Line2D]:
    handles = [
        Line2D(
            [],
            [],
            linestyle="",
            marker="s",
            markersize=10,
            markeredgecolor=CANDIDATE_COLOR,
            markerfacecolor=CANDIDATE_COLOR,
            label="Candidate",
        ),
        Line2D(
            [],
            [],
            linestyle="",
            marker="s",
            markersize=10,
            markeredgecolor=BACKGROUND_COLOR,
            markerfacecolor=BACKGROUND_COLOR,
            label="Background",
        ),
    ]
    return handles


def background_frame(df: pd.DataFrame, p_col: str) -> pd.DataFrame:
    """Eligible, non-high-priority nodes with a valid p on the given axis."""
    eligible = df[TIER_COL] != INELIGIBLE
    not_candidate = ~df[TIER_COL].isin(HIGH_PRIORITY_TIERS)
    bg = df.loc[eligible & not_candidate].copy()
    return bg[bg[p_col].notna()]


def candidate_frame(
    df: pd.DataFrame,
    candidate_tiers: str | tuple[str, ...] | list[str] | set[str],
    p_col: str,
) -> pd.DataFrame:
    """Axis-specific candidate frame."""
    eligible = df[TIER_COL] != INELIGIBLE
    candidate = df[TIER_COL].isin(_as_tier_tuple(candidate_tiers))
    cand = df.loc[eligible & candidate].copy()
    return cand[cand[p_col].notna()]

In [ ]:
# --------------------------------------------------------------------------- #
# 1. Null calibration  (the first plot to look at)
# --------------------------------------------------------------------------- #
def plot_null_calibration(
    df: pd.DataFrame,
    *,
    dimensions: tuple[str, ...] = tuple(DIMENSION),
    candidate_tiers_by_dimension: dict[str, tuple[str, ...]] | None = None,
    width: "WIDTHS" = "double",
    width_in: float | None = None,
    height_in: float = 5.8,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
    n_bins: int = 20,
) -> Figure:
    """Permutation-null calibration for local burst and onward burden axes.

    Each row is one detection axis. The left column shows upper-tail p-values;
    the right column shows observed score against the corresponding null mean.
    """
    dimensions = tuple(dimensions)
    candidate_tiers_by_dimension = (
        candidate_tiers_by_dimension or DIMENSION_CANDIDATE_TIERS
    )
    if len(dimensions) == 0:
        raise ValueError("At least one dimension is required.")

    fig, axes = new_figure(
        nrows=len(dimensions),
        ncols=2,
        sharex="col",
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
    )
    axes = axes
    panel_axes = []

    for row_idx, dimension in enumerate(dimensions):
        cols = dimension_cols(dimension)
        axis_label = dimension_label(dimension)
        candidate_tiers = candidate_tiers_by_dimension.get(
            dimension, (f"high_priority_{dimension}",)
        )

        bg = background_frame(df, cols["p"])
        cand = candidate_frame(df, candidate_tiers, cols["p"])
        p_bg = bg[cols["p"]].dropna().to_numpy()
        p_cand = cand[cols["p"]].dropna().to_numpy()

        ax_p, ax_scatter = axes[row_idx, 0], axes[row_idx, 1]
        panel_axes.extend([ax_p, ax_scatter])

        ax_p.hist(
            [p_bg, p_cand],
            bins=n_bins,
            range=(0, 1),
            stacked=True,
            color=[BACKGROUND_COLOR, CANDIDATE_COLOR],
            edgecolor="white",
            linewidth=0.4,
        )
        ax_p.axhline(
            len(p_bg) / n_bins,
            color="black",
            linestyle="--",
            linewidth=0.8,
            label="Background uniform expectation",
        )
        ax_p.set_title(axis_label)
        ax_p.set_xlabel("Composite null upper-tail $p$")
        ax_p.set_ylabel("Number of nodes")
        if row_idx == 0:
            ax_p.legend(loc="best", frameon=False)

        for frame, color, face in [
            (bg, BACKGROUND_COLOR, "none"),
            (cand, CANDIDATE_COLOR, CANDIDATE_COLOR),
        ]:
            sub = frame[[cols["null_mean"], cols["score"]]].dropna()
            ax_scatter.scatter(
                sub[cols["null_mean"]],
                sub[cols["score"]],
                s=6,
                facecolors=face,
                edgecolors=color,
                linewidths=0.4,
                alpha=0.5,
            )

        limits = df[[cols["null_mean"], cols["score"]]].to_numpy(dtype=float)
        lo = float(np.nanmin(limits))
        hi = float(np.nanmax(limits))
        ax_scatter.plot(
            [lo, hi], [lo, hi], color="black", linewidth=0.8, linestyle="--"
        )
        ax_scatter.set_title(axis_label)
        ax_scatter.set_xlabel(f"Null mean {axis_label.lower()} score")
        ax_scatter.set_ylabel(f"Observed {axis_label.lower()} score")

    fig.legend(
        handles=legend_candidate_background(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.06),
        ncol=2,
        frameon=False,
    )
    add_panel_labels(panel_axes)
    plt.close(fig)
    return fig

In [ ]:
fig = plot_null_calibration(cluster_table)
display(fig)

In [ ]:
# --------------------------------------------------------------------------- #
# 2. Volcano  (what got flagged, and how strongly)
# --------------------------------------------------------------------------- #
def plot_sse_volcano(
    df: pd.DataFrame,
    *,
    dimensions: tuple[str, ...] = tuple(DIMENSION),
    candidate_tiers_by_dimension: dict[str, tuple[str, ...]] | None = None,
    alpha_line: float = 0.05,
    width: "WIDTHS" = "double",
    width_in: float | None = None,
    height_in: float = 3.5,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
) -> Figure:
    """Volcano plots for local burst and onward burden axes.

    Each row shows null-z against -log10(upper-tail p), with point size scaled
    by cluster size. Filled markers are candidates on that axis.
    """
    dimensions = tuple(dimensions)
    candidate_tiers_by_dimension = (
        candidate_tiers_by_dimension or DIMENSION_CANDIDATE_TIERS
    )
    if len(dimensions) == 0:
        raise ValueError("At least one dimension is required.")

    fig, axes = new_figure(
        nrows=1,
        ncols=len(dimensions),
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
    )
    axes = axes

    s = 4 + 30 * np.sqrt(df["cluster_size"].clip(lower=1) / df["cluster_size"].max())

    for ax, dimension in zip(axes, dimensions):
        cols = dimension_cols(dimension)
        axis_label = dimension_label(dimension)
        candidate_tiers = candidate_tiers_by_dimension.get(
            dimension, (f"high_priority_{dimension}",)
        )

        bg = background_frame(df, cols["p"])
        cand = candidate_frame(df, candidate_tiers, cols["p"])

        for frame, color, face in [
            (bg, BACKGROUND_COLOR, "none"),
            (cand, CANDIDATE_COLOR, CANDIDATE_COLOR),
        ]:
            y = -np.log10(frame[cols["p"]].clip(lower=np.finfo(float).tiny))
            x = frame[cols["null_z"]]
            ax.scatter(
                x,
                y,
                s=s[frame.index],
                facecolors=face,
                edgecolors=color,
                linewidths=0.4,
                alpha=0.55,
            )
        ax.axhline(
            -np.log10(alpha_line),
            color="black",
            linestyle="--",
            linewidth=0.8,
            label=f"$p$ = {alpha_line:g}",
        )
        ax.set_title(axis_label)
        ax.set_xlabel(f"{axis_label} null $z$-score")
        ax.set_ylabel(r"$-\log_{10}$ upper-tail $p$")

    fig.legend(
        handles=legend_candidate_background(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=2,
        frameon=False,
    )
    add_panel_labels(axes)
    plt.close(fig)
    return fig

In [ ]:
fig = plot_sse_volcano(cluster_table)
display(fig)

In [ ]:
# --------------------------------------------------------------------------- #
# 3. Size-confounding check  (is the signal just cluster size?)
# --------------------------------------------------------------------------- #
def plot_size_confounding(
    df: pd.DataFrame,
    *,
    dimensions: tuple[str, ...] = tuple(DIMENSION),
    candidate_tiers_by_dimension: dict[str, tuple[str, ...]] | None = None,
    width: "WIDTHS" = "double",
    width_in: float | None = None,
    height_in: float = 3.5,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
) -> tuple[Figure, dict[str, float]]:
    """Null-z vs. log cluster size for each detection axis.

    Returns the figure and a Pearson-correlation dictionary keyed by axis.
    """
    dimensions = tuple(dimensions)
    candidate_tiers_by_dimension = (
        candidate_tiers_by_dimension or DIMENSION_CANDIDATE_TIERS
    )
    if len(dimensions) == 0:
        raise ValueError("At least one dimension is required.")

    fig, axes = new_figure(
        nrows=1,
        ncols=len(dimensions),
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
    )
    axes = axes
    rho_by_dimension: dict[str, float] = {}

    for ax, dimension in zip(axes, dimensions):
        cols = dimension_cols(dimension)
        axis_label = dimension_label(dimension)
        candidate_tiers = candidate_tiers_by_dimension.get(
            dimension, (f"high_priority_{dimension}",)
        )

        bg = background_frame(df, cols["p"])
        cand = candidate_frame(df, candidate_tiers, cols["p"])

        for frame, color, face in [
            (bg, BACKGROUND_COLOR, "none"),
            (cand, CANDIDATE_COLOR, CANDIDATE_COLOR),
        ]:
            ax.scatter(
                frame["log_cluster_size"],
                frame[cols["null_z"]],
                s=6,
                facecolors=face,
                edgecolors=color,
                linewidths=0.4,
                alpha=0.5,
            )

        valid = df["log_cluster_size"].notna() & df[cols["null_z"]].notna()
        rho = np.nan
        if valid.sum() >= 2:
            rho = float(
                np.corrcoef(
                    df.loc[valid, "log_cluster_size"], df.loc[valid, cols["null_z"]]
                )[0, 1]
            )
        rho_by_dimension[dimension] = rho

        ax.set_title(
            rf"{axis_label}: $r$ = {rho:.2f}", fontsize=plt.rcParams["font.size"]
        )
        ax.set_xlabel("log(1 + cluster size)")
        ax.set_ylabel(f"{axis_label} null $z$-score")

    fig.legend(
        handles=legend_candidate_background(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=2,
        frameon=False,
    )
    add_panel_labels(list(axes))
    plt.close(fig)
    return fig, rho_by_dimension

In [ ]:
fig, rho_by_dimension = plot_size_confounding(cluster_table)
display(fig)
print("Pearson correlation between log cluster size and null z-score:")
for dimension, rho in rho_by_dimension.items():
    print(f"  {dimension}: {rho:.3f}")

In [ ]:
# --------------------------------------------------------------------------- #
# 4. Component contributions  (which channels drive flagging)
# --------------------------------------------------------------------------- #
def plot_component_contributions(
    df: pd.DataFrame,
    *,
    ranked_columns: tuple[str, ...] | list[str] | None = None,
    column_labels: dict[str, str] | None = None,
    candidate_tiers: str
    | tuple[str, ...]
    | list[str]
    | set[str] = DEFAULT_CANDIDATE_TIERS,
    tier_col: str = TIER_COL,
    width: "WIDTHS" = "double",
    width_in: float | None = None,
    height_in: float = 4.8,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
    size_ineligible: str = INELIGIBLE,
) -> Figure:
    """Ranked component distributions for candidates vs background."""
    ranked_columns = (
        "log_cluster_size",
        "sampling_adjusted_excess_size",
        "log_excess_over_upstream",
        "log_new_downstream_burden_ratio",
    )
    column_labels = {
        "sampling_adjusted_excess_size": "Context-adjusted excess size",
        "log_cluster_size": "Cluster size",
        "log_excess_over_upstream": "Excess over upstream",
        "log_new_downstream_burden_ratio": "New downstream burden ratio",
        **(column_labels or {}),
    }

    df = df.loc[df[tier_col] != size_ineligible].copy()
    is_cand = _candidate_mask(df, tier_col=tier_col, candidate_tiers=candidate_tiers)

    plot_columns: list[tuple[str, str]] = []
    missing: list[str] = []
    for col in ranked_columns:
        base_col = col.removesuffix("_pct_window")
        rank_col = col if col.endswith("_pct_window") else f"{col}_pct_window"
        if rank_col not in df.columns and base_col in df.columns:
            if "window_idx" in df.columns:
                df[rank_col] = df.groupby("window_idx", dropna=False)[base_col].rank(
                    pct=True,
                    method="average",
                )
            else:
                df[rank_col] = df[base_col].rank(pct=True, method="average")
        if rank_col in df.columns:
            plot_columns.append(
                (
                    rank_col,
                    column_labels.get(base_col, base_col.replace("_", " ").title()),
                )
            )
        else:
            missing.append(col)

    if not plot_columns:
        raise ValueError(f"None of the requested ranked columns are present: {missing}")

    fig, axes = new_figure(
        nrows=2,
        ncols=2,
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
        sharex=True,
        # sharey=True,
    )
    axes = axes.ravel()

    for ax, (col, label) in zip(axes, plot_columns):
        for mask, color, fill in [
            (~is_cand, BACKGROUND_COLOR, False),
            (is_cand, CANDIDATE_COLOR, True),
        ]:
            vals = df[mask][col].dropna().to_numpy()
            if vals.size < 5:
                continue
            ax.hist(
                vals,
                bins=25,
                range=(0, 1),
                density=True,
                histtype="stepfilled" if fill else "step",
                color=color,
                alpha=0.5 if fill else 1.0,
                edgecolor=color,
                linewidth=1.0,
            )
        ax.set_title(label)
        ax.set_xlim(0, 1)
        ax.set_yticks([])
    for ax in axes[[0, 2]]:
        ax.set_ylabel("Density")

    for ax in axes[[2, 3]]:
        ax.set_xlabel("Within-window percentile rank")

    handles = [
        Line2D(
            [], [], color=CANDIDATE_COLOR, linewidth=4, alpha=0.5, label="Candidate"
        ),
        Line2D([], [], color=BACKGROUND_COLOR, linewidth=1.5, label="Background"),
    ]

    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=2,
        frameon=False,
    )
    add_panel_labels(list(axes))
    plt.close(fig)
    return fig


In [ ]:
fig = plot_component_contributions(cluster_table)
display(fig)

In [ ]:
# --------------------------------------------------------------------------- #
# 5. SSE phenotype  (burst vs onward burden)
# --------------------------------------------------------------------------- #
def plot_sse_phenotype(
    node_stats: pd.DataFrame,
    *,
    burst_col: str = "burst_score",
    burden_col: str = "burden_score",
    size_col: str = "cluster_size",
    tier_col: str = "candidate_tier",
    dimensions: tuple[str, ...] = tuple(DIMENSION),
    candidate_tiers_by_dimension: dict[str, tuple[str, ...]] | None = None,
    width: "WIDTHS" = "double",
    width_in: float | None = None,
    height_in: float = 3.5,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
    size_ineligible: str = INELIGIBLE,
) -> Figure:
    """Burst-vs-burden space, split by the current detection-axis candidates."""
    df = node_stats.loc[node_stats[tier_col] != size_ineligible].copy()
    candidate_tiers_by_dimension = (
        candidate_tiers_by_dimension or DIMENSION_CANDIDATE_TIERS
    )
    dimensions = tuple(dimensions)
    if burst_col not in df.columns or burden_col not in df.columns:
        raise KeyError(f"Missing `{burst_col}` or `{burden_col}` in node_stats.")

    fig, axes = new_figure(
        nrows=1,
        ncols=2,
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
        sharex=True,
        sharey=True,
    )
    axes = axes
    s = 4 + 30 * np.sqrt(df[size_col].clip(lower=1) / df[size_col].max())

    for ax, dimension in zip(axes, dimensions):
        axis_label = dimension_label(dimension)
        is_cand = _candidate_mask(
            df,
            tier_col=tier_col,
            candidate_tiers=candidate_tiers_by_dimension.get(
                dimension, (f"high_priority_{dimension}",)
            ),
        )
        for mask, color, face in [
            (~is_cand, BACKGROUND_COLOR, "none"),
            (is_cand, CANDIDATE_COLOR, CANDIDATE_COLOR),
        ]:
            ax.scatter(
                df.loc[mask, burst_col],
                df.loc[mask, burden_col],
                s=s[mask],
                facecolors=face,
                edgecolors=color,
                linewidths=0.4,
                alpha=0.55,
            )
        ax.axvline(
            df[burst_col].median(skipna=True), color=GRAY, linewidth=0.6, linestyle=":"
        )
        ax.axhline(
            df[burden_col].median(skipna=True), color=GRAY, linewidth=0.6, linestyle=":"
        )
        ax.set_title(axis_label)
        ax.set_ylabel("Onward burden score")
        ax.set_xlabel("Local burst score")

    fig.legend(
        handles=legend_candidate_background(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=2,
        frameon=False,
    )
    add_panel_labels(list(axes))
    plt.close(fig)
    return fig

In [ ]:
fig = plot_sse_phenotype(cluster_table)
display(fig)

In [ ]:
# --------------------------------------------------------------------------- #
# 6. Temporal context  (candidates over epidemic time)
# --------------------------------------------------------------------------- #
def plot_temporal_candidates(
    node_stats: pd.DataFrame,
    *,
    window_col: str = "window_idx",
    tier_col: str = "candidate_tier",
    dimensions: tuple[str, ...] = tuple(DIMENSION),
    candidate_tiers_by_dimension: dict[str, tuple[str, ...]] | None = None,
    epidemic_col: str | None = "wn_positive_tests",
    width: "WIDTHS" = "double",
    width_in: float | None = None,
    height_in: float = 5.2,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
    size_ineligible: str = INELIGIBLE,
) -> tuple[Figure, pd.DataFrame]:
    """Candidate count per window, split by detection axis."""
    df = node_stats.loc[node_stats[tier_col] != size_ineligible].copy()
    candidate_tiers_by_dimension = (
        candidate_tiers_by_dimension or DIMENSION_CANDIDATE_TIERS
    )
    dimensions = tuple(dimensions)

    summaries = []
    for dimension in dimensions:
        is_cand = _candidate_mask(
            df,
            tier_col=tier_col,
            candidate_tiers=candidate_tiers_by_dimension.get(
                dimension, (f"high_priority_{dimension}",)
            ),
        )
        summary = (
            df.assign(
                _cand=is_cand,
                dimension=dimension,
                dimension_label=dimension_label(dimension),
            )
            .groupby(["dimension", "dimension_label", window_col], dropna=False)
            .agg(n_nodes=("_cand", "size"), n_candidates=("_cand", "sum"))
            .reset_index()
            .sort_values(window_col)
        )
        summary["candidate_rate"] = summary["n_candidates"] / summary["n_nodes"]
        summaries.append(summary)
    summary_all = pd.concat(summaries, ignore_index=True)

    fig, axes = new_figure(
        nrows=len(dimensions),
        ncols=1,
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
        sharex=True,
    )
    axes = axes

    epi = None
    if epidemic_col is not None and epidemic_col in df.columns:
        epi = (
            df.groupby(window_col, dropna=False)[epidemic_col]
            .first()
            .reset_index()
            .sort_values(window_col)
        )

    for ax, dimension in zip(axes, dimensions):
        axis_label = dimension_label(dimension)
        sub = summary_all.loc[summary_all["dimension"].eq(dimension)]
        ax.bar(
            sub[window_col],
            sub["n_candidates"],
            color=CANDIDATE_COLOR,
            alpha=0.85,
            width=0.9,
            label="Candidate nodes",
        )
        ax.set_title(axis_label)
        ax.set_ylabel("Candidate nodes", color=CANDIDATE_COLOR)
        ax.tick_params(axis="y", labelcolor=CANDIDATE_COLOR)

        if epi is not None:
            ax2 = ax.twinx()
            ax2.plot(
                epi[window_col],
                epi[epidemic_col],
                color="black",
                linewidth=1.2,
                label=epidemic_col,
            )
            ax2.set_ylabel("Window positive tests", color="black")
            ax2.set_ylim(bottom=0)

    axes[-1].set_xlabel("Window index")
    add_panel_labels(list(axes))
    plt.close(fig)
    return fig, summary_all

In [ ]:
fig, df1 = plot_temporal_candidates(cluster_table)
display(fig)

In [ ]:
# --------------------------------------------------------------------------- #
# 7. Candidate rate by clade / VOC
# --------------------------------------------------------------------------- #
def plot_candidate_rate_by_clade(
    node_stats: pd.DataFrame,
    *,
    clade_col: str = "clade",
    tier_col: str = "candidate_tier",
    dimensions: tuple[str, ...] = tuple(DIMENSION),
    candidate_tiers_by_dimension: dict[str, tuple[str, ...]] | None = None,
    # min_group_n: int = 20,
    width: "WIDTHS" = "double",
    width_in: float | None = None,
    height_in: float = 3.5,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
    size_ineligible: str = INELIGIBLE,
) -> tuple[Figure, pd.DataFrame]:
    """Candidate proportion by group, split by detection axis."""
    df = node_stats.loc[node_stats[tier_col] != size_ineligible].copy()
    candidate_tiers_by_dimension = (
        candidate_tiers_by_dimension or DIMENSION_CANDIDATE_TIERS
    )
    dimensions = tuple(dimensions)
    df[clade_col] = df[clade_col].map(CLADES).fillna("Other")

    records = []
    for dimension in dimensions:
        is_cand = _candidate_mask(
            df,
            tier_col=tier_col,
            candidate_tiers=candidate_tiers_by_dimension.get(
                dimension, (f"high_priority_{dimension}",)
            ),
        )
        tab = (
            df.assign(_cand=is_cand.astype(int))
            .groupby(clade_col, dropna=False)
            .agg(n=("_cand", "size"), k=("_cand", "sum"))
            .reset_index()
        )
        
        # tab = tab.loc[tab["n"] >= min_group_n].copy()
        tab["rate"] = tab["k"] / tab["n"]

        z = 1.96
        n, p = tab["n"].to_numpy(), tab["rate"].to_numpy()
        denom = 1 + z**2 / n
        centre = (p + z**2 / (2 * n)) / denom
        half = (z / denom) * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2))
        tab["lo"], tab["hi"] = centre - half, centre + half
        tab["overall_rate"] = float(is_cand.mean())
        tab["excludes_overall"] = (tab["lo"] > tab["overall_rate"]) | (
            tab["hi"] < tab["overall_rate"]
        )
        tab["dimension"] = dimension
        tab["dimension_label"] = dimension_label(dimension)
        records.append(tab)

    tab_all = pd.concat(records, ignore_index=True)
    fig, axes = new_figure(
        nrows=1,
        ncols=2,
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
        sharex=True,
        sharey=True,
    )
    axes = axes

    for ax, dimension in zip(axes, dimensions):
        tab = (
            tab_all.loc[tab_all["dimension"].eq(dimension)]
            .sort_values("rate", ascending=True)
            .reset_index(drop=True)
        )
        y = np.arange(len(tab))
        for yi, (_, r) in zip(y, tab.iterrows()):
            ax.plot(
                [r["lo"], r["hi"]], [yi, yi], color="black", linewidth=0.9, zorder=1
            )
            ax.scatter(
                r["rate"],
                yi,
                s=28,
                zorder=2,
                facecolors=CANDIDATE_COLOR if r["excludes_overall"] else "none",
                edgecolors=CANDIDATE_COLOR,
                linewidths=1.0,
            )
        overall = float(tab["overall_rate"].iloc[0]) if len(tab) else 0.0
        ax.axvline(
            overall, color=GRAY, linestyle="--", linewidth=0.8, label="Overall rate"
        )
        ax.set_title(dimension_label(dimension))
        ax.set_yticks(y)
        ax.set_yticklabels(tab[clade_col].astype(str))
        ax.legend(loc="lower right", frameon=False)
        ax.set_xlabel("Candidate proportion (Wilson 95% CI)")
    axes[0].set_ylabel("Clade")
    fig.legend(
        handles=legend_candidate_background(),
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=2,
        frameon=False,
    )
    add_panel_labels(list(axes))
    plt.close(fig)
    return fig, tab_all

In [ ]:
fig, df2 = plot_candidate_rate_by_clade(cluster_table)
display(fig)

In [ ]:
# --------------------------------------------------------------------------- #
# 8. Candidate tier by graph role
# --------------------------------------------------------------------------- #
def plot_tier_by_graph_role(
    node_stats: pd.DataFrame,
    *,
    role_col: str = "primary_graph_role",
    tier_col: str = "candidate_tier",
    dimensions: tuple[str, ...] = tuple(DIMENSION),
    candidate_tiers_by_dimension: dict[str, tuple[str, ...]] | None = None,
    exclude_censored: bool = True,
    width: "WIDTHS" = "onehalf",
    width_in: float | None = None,
    height_in: float = 5.6,
    context: "CONTEXTS" = "paper",
    font_scale: float = 1.0,
    size_ineligible: str = INELIGIBLE,
) -> tuple[Figure, pd.DataFrame]:
    """Candidate proportion by primary graph role, split by detection axis."""
    required_cols = [tier_col, role_col]
    missing = [col for col in required_cols if col not in node_stats.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = node_stats.loc[node_stats[tier_col] != size_ineligible].copy()
    if df.empty:
        raise ValueError("No size-eligible nodes available for plotting.")

    if exclude_censored:
        keep = pd.Series(True, index=df.index)
        for col in ("left_censored", "right_censored", "boundary_censored"):
            if col in df.columns:
                keep &= ~df[col].fillna(False).astype(bool)
        df = df.loc[keep].copy()
    if df.empty:
        raise ValueError("No nodes remain after applying censoring exclusions.")

    df[role_col] = df[role_col].fillna("unknown").astype("string")
    candidate_tiers_by_dimension = (
        candidate_tiers_by_dimension or DIMENSION_CANDIDATE_TIERS
    )
    dimensions = tuple(dimensions)

    role_labels = {
        "isolated": "Isolated",
        "single_outgoing_source": "Single-outgoing source",
        "source_branching": "Source branching",
        "single_incoming_sink": "Single-incoming sink",
        "merging_sink": "Merging sink",
        "linear_continuation": "Linear continuation",
        "internal_branching": "Internal branching",
        "internal_merging": "Internal merging",
        "merge_and_branch": "Merge + branch",
        "other": "Other",
        "unknown": "Unknown",
    }
    role_order = [
        "isolated",
        "single_outgoing_source",
        "source_branching",
        "linear_continuation",
        "internal_branching",
        "internal_merging",
        "merge_and_branch",
        "merging_sink",
        "single_incoming_sink",
        "other",
        "unknown",
    ]

    tabs = []
    for dimension in dimensions:
        is_candidate = _candidate_mask(
            df,
            tier_col=tier_col,
            candidate_tiers=candidate_tiers_by_dimension.get(
                dimension, (f"high_priority_{dimension}",)
            ),
        )
        tab = (
            df.assign(_is_candidate=is_candidate.astype(bool))
            .groupby(role_col, dropna=False)
            .agg(n=("_is_candidate", "size"), k=("_is_candidate", "sum"))
            .reset_index()
            .rename(columns={role_col: "primary_graph_role"})
        )
        tab["rate"] = tab["k"] / tab["n"]
        tab["role_label"] = (
            tab["primary_graph_role"]
            .astype(str)
            .map(role_labels)
            .fillna(tab["primary_graph_role"].astype(str))
        )
        tab["_role_order"] = pd.Categorical(
            tab["primary_graph_role"].astype(str),
            categories=role_order,
            ordered=True,
        )
        tab = tab.sort_values(["_role_order", "primary_graph_role"]).drop(
            columns="_role_order"
        )
        tab["overall_rate"] = float(is_candidate.mean())
        tab["dimension"] = dimension
        tab["dimension_label"] = dimension_label(dimension)
        tabs.append(tab.reset_index(drop=True))
    tab_all = pd.concat(tabs, ignore_index=True)

    fig, axes = new_figure(
        nrows=len(dimensions),
        ncols=1,
        width=width,
        width_in=width_in,
        height_in=height_in,
        context=context,
        font_scale=font_scale,
        layout="constrained",
        sharex=True,
    )
    axes = axes
    ymax = max(0.05, min(1.0, tab_all["rate"].max() * 1.18))

    for row_idx, (ax, dimension) in enumerate(zip(axes, dimensions)):
        tab = tab_all.loc[tab_all["dimension"].eq(dimension)].reset_index(drop=True)
        x = np.arange(len(tab))
        ax.bar(x, tab["rate"], color=CANDIDATE_COLOR, alpha=0.85, width=0.7)
        ax.axhline(
            float(tab["overall_rate"].iloc[0]),
            color=GRAY,
            linestyle="--",
            linewidth=0.8,
            label="Overall tested-node rate",
        )
        for xi, row in zip(x, tab.itertuples(index=False)):
            ax.text(
                xi,
                row.rate,
                f"{row.k:,}/{row.n:,}",
                ha="center",
                va="bottom",
                fontsize=plt.rcParams["font.size"] * 0.75,
            )
        ax.set_title(dimension_label(dimension))
        ax.set_ylabel("Candidate proportion")
        ax.set_ylim(0, ymax)
        ax.set_xticks(x)
        ax.set_xticklabels(tab["role_label"], rotation=30, ha="right")
        ax.legend(loc="upper left", frameon=False)
        if row_idx < len(axes) - 1:
            ax.tick_params(axis="x", labelbottom=False)

    add_panel_labels(list(axes))
    plt.close(fig)
    return fig, tab_all

In [ ]:
fig, df3 = plot_tier_by_graph_role(cluster_table)
display(fig)